# 2、工具的定义方式1：不使用@tool
## 2.1 模型绑定工具并发送请求

In [3]:
from langchain_community.chat_models import ChatZhipuAI
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = ChatZhipuAI(
    model="glm-5.3-flash",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)

python-dotenv could not parse statement starting at line 17


In [5]:
#声明一个函数
def get_weather(city : str):
    return f'{city}天气晴朗'

#将工具绑定在模型上
model_with_tools = model.bind_tools([get_weather])

#调用模型
response = model_with_tools.invoke('北京天气怎么样')
rprint(response)

AIMessage(
    content='',
    additional_kwargs={
        'tool_calls': [
            {
                'function': {'arguments': '{"city":"北京"}', 'name': 'get_weather'},
                'id': 'call_-7166699906051077806',
                'index': 0,
                'type': 'function'
            }
        ]
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 37,
            'completion_tokens_details': {'reasoning_tokens': 25},
            'prompt_tokens': 150,
            'prompt_tokens_details': {'cached_tokens': 0},
            'total_tokens': 187
        },
        'model_name': 'glm-5.3-flash',
        'finish_reason': 'tool_calls'
    },
    id='lc_run--01a13ea8-3e22-7852-b1be-2038296872ba-0',
    tool_calls=[
        {'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_-7166699906051077806', 'type': 'tool_call'}
    ],
    invalid_tool_calls=[]
)

# 2.2 工具描述的各部分详解

## 2.2.1 了解：convert_to_openai_tool

执行 model.bind_tools([get_weather])，底层最终会调用 convert_to_openai_tool 生成工具描述。所以我们可以直接调用后者查看解析后的工具描述。 

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city : str):
    return f'{city}天气晴朗'

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.2 description说明

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city : str):
    '''天气查询工具'''
    return f'{city}天气晴朗'

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

 ## 2.3 参数说明

 注意换行

In [13]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city : str):
    '''
    天气查询工具

    Args:
        city:城市名称
    
    Returns:
        返回城市的天气
    '''

    return f'{city}天气晴朗'

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {'city': {'description': '城市名称', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.4 参数类型的说明
举例一

In [15]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city):
    '''
    天气查询工具
    '''

    return f'{city}天气晴朗'

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2
如果在docstring中声明了参数的描述，则必须在函数声明处指明参数类型

In [20]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city):
    '''
    天气查询工具

    Args:
        city:具体的城市
    '''

    return f'{city}天气晴朗'

rprint(convert_to_openai_tool(get_weather))

ValueError: Arg city in docstring not found in function signature.

## 2.5 参数默认值的说明
一旦参数设置的默认值，则打印的结果中的required字段中就不再包含此参数

举例1

In [24]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(dt:str,city:str = 'beijing'):
    '''
    天气查询工具

    Args:
        city:具体的城市
        dt:时间
    '''

    return f'{city}天气晴朗'

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {
                'dt': {'description': '时间', 'type': 'string'},
                'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}